## Preprocesamiento y Modelado

Una vez inspeccionado el dataset en `customer_churn_eda.ipynb` definimos una una estrategia de preprocesamiento iterativo (de menos a más) para el encontrar

In [151]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, make_scorer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.discriminant_analysis import StandardScaler


### Configuración de constantes, rutas y variables 

En esta sección definimos constantes, rutas de archivos y atributos del dataset

In [152]:
# Rutas de los archivos de datos
TRAIN_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/train.csv"
TEST_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Cargamos los datos
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Definir variables objetivo
TARGET = 'Exited'
# Variables numéricas: Incluyo las continuas y las binarias numéricas (HasCrCard, IsActiveMember)
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
# Variables categóricas: Las de texto con pocas categorías
CAT_FEATURES = ['Geography', 'Gender']
# Variables a eliminar inicialmente (IDs y apellido)
DROP_FEATURES = ['CustomerId', 'Surname']
SURNAME_COL = 'Surname'
RANDOM_STATE = 100            # Semilla para reproducibilidad

### 1. Preparación de Datos

Separamos variables independientes y dependientes en X_train e y_train por convención.

In [153]:
# X_train = variables independientes
# y_train = variable dependiente u objetivo
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

---------------------------------------

In [154]:
def make_preprocessor(X: pd.DataFrame, version: str,numerical_cols:[], categorical_cols:[]):
    """Genera un ColumnTransformer con pipelines de preprocesamiento
    para variables numéricas y categóricas según la versión indicada.

    Args:
        X (pd.DataFrame): _input data frame_
        version (str): Versión del preprocesamiento en formato 'N#_C#'
        e.g. 'N3_C1' donde N# indica la versión numérica y C# la categórica
        numerical_cols (_type_): columnas numéricas para el preprocesamiento
        categorical_cols (_type_): columnas categóricas para el preprocesamiento

    Raises:
        ValueError: _unknown numeric version_
        ValueError: _unknown categorical version_

    Returns:
        _type_: ColumnTransformer con pipelines de preprocesamiento
    """
    num_version, categorical_version = version.split("_")  # e.g. 'N3', 'C1'

    # --- Numerical pipeline ---
    numerical_transformers = []

    if num_version == "N1":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N2":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana + indicador de faltantes
        # add_indicator=True añade una columna booleana por cada numérica
        # que indica si el valor estaba faltante
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    elif num_version == "N3":
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana + indicador de faltantes + escalado estandar
        num_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ])
        numerical_transformers.append(("num", num_pipe, numerical_cols))
    else:
        raise ValueError(f"Unknown numeric version: {num_version}")

    # --- Categorical pipeline (si aplica) ---
    categorical_transformers = []
    if categorical_version == "C0": 
        # No aplica pipeline en categoricas
        pass
    elif categorical_version == "C1":
        # Categóricas normales: imputar + onehot
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
        categorical_transformers.append(("cat", categorical_pipe, categorical_cols))

    else:
        raise ValueError(f"Unknown categorical version: {categorical_version}")

    # --- Build ColumnTransformer ---
    # Transformers numéricos y categóricos
    transformers = []
    transformers.extend(numerical_transformers)     
    transformers.extend(categorical_transformers)

    # Construimos el ColumnTransformer final
    # que une pipelines numéricos + pipelines categóricos
    col_trans_preprocessor = ColumnTransformer(
        transformers=transformers, # lista de tuplas (name, pipeline, cols)
        remainder="drop", # elimina columnas no especificadas
        verbose_feature_names_out=True) # nombres detallados de columnas
    return col_trans_preprocessor


In [155]:
# Creación y evaluación del pipeline
# Construcción del pipeline con preprocesador y modelo

def make_pipeline(preprocessor: ColumnTransformer, model=None):
    """Construye un Pipeline con el preprocesador y el modelo indicado.
    Args:
        preprocessor (ColumnTransformer): Preprocesador ColumnTransformer
        model (_type_, optional): Modelo de clasificación. Defaults to None.
    Returns:
        Pipeline: Pipeline con preprocesador y modelo, si no se indica modelo
        se usa LinearDiscriminantAnalysis por defecto.
    """
    if model is None:
        model = LinearDiscriminantAnalysis()
        #model = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])

def evaluate_pipeline(pipe: Pipeline, X: pd.DataFrame, y: pd.Series, n_splits=5):
    """ Evalúa el pipeline usando Validación Cruzada estratificada
    Args:
        pipe (Pipeline): Pipeline a evaluar.
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        dict: Diccionario con las métricas promedio y desviación estándar.
    """
    # Configuramos la Validación Local Cruzada  n_splits splits (divisiones)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    # Definimos las métricas que queremos extraer
    # f1, roc_auc, precision, recall, accuracy son strings estándar de sklearn.
    # Kappa requiere make_scorer.
    scoring_metrics = {
        "f1": "f1",
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        'kappa': make_scorer(cohen_kappa_score),
        'precision': 'precision',
        'recall': 'recall',
    }
    cv_results = cross_validate(pipe, X, y, cv=cv, scoring=scoring_metrics, n_jobs=-1)
    return {
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std":  cv_results["test_f1"].std(),
        "auc_mean": cv_results["test_roc_auc"].mean(),
        "auc_std":  cv_results["test_roc_auc"].std(),
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std":  cv_results["test_accuracy"].std(),
        "kappa_mean": cv_results["test_kappa"].mean(),
        "kappa_std":  cv_results["test_kappa"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std":  cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std":  cv_results["test_recall"].std()
    }


In [156]:

def benchmark_models_with_fixed_preprocess(X: pd.DataFrame, y: pd.Series, models: dict, 
                                           best_preprocesor_version: str, n_splits=5):
    """Evalúa varios modelos con un preprocesador fijo usando Validación Cruzada.
    Args:
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        models (dict): Diccionario con nombre y objeto del modelo a evaluar.
        best_preprocesor_version (str): Versión del preprocesador a usar.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        pd.DataFrame: DataFrame con resultados de cada modelo evaluado.
    """
    # Construimos el preprocesador fijo con la mejor versión
    best_preprocesor = make_preprocessor(X, best_preprocesor_version,NUM_FEATURES,CAT_FEATURES)

    model_results = []  # Lista para almacenar resultados de cada modelo
    # Evaluamos cada modelo con el preprocesador fijo
    for name, model in models.items():
        # Construimos el pipeline con preprocesador fijo y el modelo actual
        pipe = Pipeline([("preprocessor", best_preprocesor), ("classifier", model)])
        try:
            # Evaluamos el pipeline con validación cruzada para el pipeline actual
            cv_metrics = evaluate_pipeline(pipe, X, y , n_splits=n_splits)
            model_results.append({
                "model": name,
                "preprocessor_version": best_preprocesor_version,
                **cv_metrics
            })
        except Exception as e:
            model_results.append({"model": name, "error": str(e)})
    # Construimos el DataFrame de resultados ordenado por F1 medio
    out = pd.DataFrame(model_results).sort_values(by="f1_mean", ascending=False, na_position="last")
    return out


------------------
## Evaluación de diferentes preprocesadores y modelos

A partir de aquí comenzamos la evaluación de los distintos preprocesadores que se han configurado en la función `make_preprocesor` y modelos, configurador `benchmark_models_with_fixed_preprocess`

In [157]:
EXPERIMENTS = [
    "N1_C0",  # num only, simple imputer
    "N2_C0",  # num only, simple imputer + indicator
    "N3_C0",  # num only, simple imputer + indicator + scaler
    "N3_C1",  # num + cat (onehot) sin surname
]

# Modelos: empieza simple
models = {
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"),
}


Evaluamos las diferentes configuraciones de preprocesador que tenemos configuradas con el modelo por defecto definido en la función `make_pipeline` con el objetivo de obtener mejor versión o combinación de preprocesado. 

In [158]:
preprocesors_results = []
for experiment_version in EXPERIMENTS:
    best_preprocesor = make_preprocessor(train_df, experiment_version,NUM_FEATURES,CAT_FEATURES)
    pipe = make_pipeline(best_preprocesor) # Construimos el pipeline con el modelo por defecto
    cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
    preprocesors_results.append({"version": experiment_version, **cv_metrics})
    #print(experiment_version, metrics)

results_df = pd.DataFrame(preprocesors_results).sort_values("f1_mean", ascending=False)
display(results_df)
best_preprocesor_version = results_df.iloc[0]["version"]
print("Mejor versión de preprocesador:", best_preprocesor_version)

,version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
3,N3_C1,0.328859,0.016720,0.768109,0.008109,0.80725,0.005654,0.238822,0.018148,0.568047,0.037219,0.231902,0.014596
1,N2_C0,0.267752,0.023950,0.747878,0.010841,0.80200,0.004444,0.186025,0.023459,0.542068,0.033285,0.177914,0.017781
2,N3_C0,0.267752,0.023950,0.747878,0.010841,0.80200,0.004444,0.186025,0.023459,0.542068,0.033285,0.177914,0.017781
0,N1_C0,0.253213,0.025006,0.747821,0.010336,0.79975,0.004978,0.172250,0.024688,0.526555,0.040729,0.166871,0.018364


Mejor versión de preprocesador: N3_C1


Evaluamos los modelos con el mejor preprocesador encontrado 

In [159]:
# Evaluamos los modelos con el mejor preprocesador encontrado
models_df = benchmark_models_with_fixed_preprocess(X_train, y_train, models, best_preprocesor_version, n_splits=5)
print("---- Resultados de validación cruzada de modelos con preprocesador fijo:----")
display(models_df)

best_model_name = models_df.iloc[0]["model"]
print("Mejor versión de preprocesador:", best_preprocesor_version)
print("Mejor modelo:", best_model_name)

---- Resultados de validación cruzada de modelos con preprocesador fijo:----


,model,preprocessor_version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
1,LogisticRegression,N3_C1,0.498472,0.011448,0.769012,0.008504,0.715125,0.005624,0.321069,0.014500,0.388648,0.007686,0.695092,0.023966
0,LinearDiscriminantAnalysis,N3_C1,0.328859,0.016720,0.768109,0.008109,0.807250,0.005654,0.238822,0.018148,0.568047,0.037219,0.231902,0.014596


Mejor versión de preprocesador: N3_C1
Mejor modelo: LogisticRegression


## Construcción del pipeline final para Kaggle

Una vez obtenido la mejor combinación de preprocesadores y el mejor modelo, construimos el pipeline final para kaggle con la mejor combinación de ambos y volvemos a ejecutar la evaluación y el entrenamiento para finalmente obtener la predicción y generar el fichero para kaggle. 

In [ ]:
# Construcción del pipeline final para Kaggle con el mejor preprocesador y modelo
# Construimos el preprocesador fijo con la mejor versión
best_preprocesor = make_preprocessor(X_train, best_preprocesor_version,NUM_FEATURES,CAT_FEATURES)
best_model = models[best_model_name]
# Pipeline Completo (Preprocesamiento + Modelo)
best_model_pipeline = Pipeline(steps=[
    ('preprocessor', best_preprocesor),
    ('classifier', best_model)
])
# Configuramos y ejecutamos la Validación Cruzada local
cv_metrics = evaluate_pipeline(best_model_pipeline, X_train, y_train, n_splits=5)
# Generación de Submission para Kaggle con el mejor modelo encontrado
# Re-entrenamos con TODOS los datos de train para la predicción final
best_model_pipeline.fit(X_train, y_train) 
test_predictions = best_model_pipeline.predict(test_df)

# Crear fichero de salida
submission_df = pd.DataFrame({
    'CustomerId': test_df['CustomerId'],
    'Exited': test_predictions
})
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Fichero '{SUBMISSION_PATH}' generado correctamente.")

print("---- Mejores Resultados y Validación Cruzada local -----")
print("Mejor modelo:", best_model_name)
print("Mejor versión de preprocesador:", best_preprocesor_version)
print(f"Mean F1-Score:  {cv_metrics['f1_mean']:.4f} (+/- Std {cv_metrics['f1_std']:.4f})")
print(f"Mean Accuracy:  {cv_metrics['accuracy_mean']:.4f} (+/- Std {cv_metrics['accuracy_std']:.4f})")
print(f"Mean Kappa:     {cv_metrics['kappa_mean']:.4f}")
print(f"Mean Precision: {cv_metrics['precision_mean']:.4f}")
print(f"Mean Recall:    {cv_metrics['recall_mean']:.4f}")



Fichero '/kaggle/working/submission.csv' generado correctamente.
Mejor modelo: LogisticRegression
Mejor versión de preprocesador: N3_C1
---- Resultados de Validación Cruzada: -----
Mean F1-Score:  0.4985 (+/- Std 0.0114)
Mean Accuracy:  0.7151 (+/- Std 0.0056)
Mean Kappa:     0.3211
Mean Precision: 0.3886
Mean Recall:    0.6951
